In [52]:
import random
from tqdm import tqdm
from sklearn.model_selection import train_test_split

import pandas as pd
import numpy as np
import os

In [53]:
seed = 42
np.random.seed(seed)

In [54]:
df = pd.read_csv("2012-2013-data-with-predictions-4-final.csv", low_memory=False, encoding="ISO-8859-1")
df.dropna(inplace=True, subset=['skill', 'problem_id'])
df.head()

,problem_log_id,skill,problem_id,user_id,assignment_id,assistment_id,start_time,end_time,problem_type,original,...,overlap_time,template_id,answer_id,answer_text,first_action,problemlogid,Average_confidence(FRUSTRATED),Average_confidence(CONFUSED),Average_confidence(CONCENTRATING),Average_confidence(BORED)
1,138083797,Rounding,365981,61394,573819,204043,2012-10-09 11:01:52,2012-10-09 11:02:13.182,algebra,1,...,21175,204043,NaN,74.29,0,138083797,0.361323,0.0,0.766925,0.000000
2,142332619,Multiplication and Division Integers,426415,61394,734130,247525,2013-03-07 10:53:20,2013-03-07 10:53:28.661,algebra,1,...,8645,247525,NaN,00,0,142332619,0.361323,0.0,0.766925,0.442968
3,145939397,Proportion,86686,61394,821352,48081,2013-08-20 19:54:56,2013-08-20 19:55:21.753,algebra,1,...,25728,46362,NaN,3.8,0,145939397,0.775000,0.0,0.766925,0.912281
5,140218191,Exponents,401234,76592,639711,228996,2012-12-12 21:00:55,2012-12-12 21:01:07.536,algebra,1,...,12522,129499,NaN,1024,0,140218191,0.361323,0.0,0.766925,0.000000
7,138367797,Equation Solving Two or Fewer Steps,87699,78401,581453,49019,2012-10-16 10:30:54,2012-10-16 10:31:55.445,algebra,1,...,61439,46279,NaN,-4,0,138367797,0.361323,0.0,0.766925,0.000000


In [55]:
df = df[['user_id', 'problem_id', 'skill', 'problem_type', 'ms_first_response', 'attempt_count', 'correct']]
df.to_csv('assist.csv', index=False)

In [57]:
key = 'problem_id'

key_q = 'q_idx'
key_s = 's_idx'
key_qt = 'q_type'
key_qd = 'q_diff'
key_ms_first_response = 'ms_first_response'
key_attempts = 'attempt_count'

question_id_dict = dict(zip(df[key].unique(), range(len(df[key].unique()))))
skill_id_dict = dict(zip(df['skill'].unique(), range(len(df['skill'].unique()))))
question_type_dict = dict(zip(df['problem_type'].unique(), range(len(df['problem_type'].unique()))))
user_id_dict = dict(zip(df['user_id'].unique(), range(len(df['user_id'].unique()))))

# question idx
df[key_q] = df[key].map(question_id_dict)
df[key_s] = df['skill'].map(skill_id_dict)
df[key_qt] = df['problem_type'].map(question_type_dict)


n_question = len(question_id_dict)
n_skill = len(skill_id_dict)
n_question_type = len(question_type_dict)
n_user = len(user_id_dict)

print("n_user:{}, num of question:{}, num of skill:{}, n_question_type:{}".format(n_user, n_question, n_skill, n_question_type))

n_user:28834, num of question:50988, num of skill:198, n_question_type:6


In [58]:
group1 = df[[key, 'correct']].groupby([key]).apply(lambda r:r['correct'].sum() / len(r['correct']))
group1
df[key_qd] = df['problem_id'].map(group1)

In [59]:
# 我们需要将数据进行预处理，每个学生的学习记录利用group by合并为序列。
group = df[['user_id', key_q, key_s, key_qt, key_qd,key_ms_first_response, key_attempts, 'correct']].groupby(['user_id']).apply(lambda r: (
            r[key_q].values,
            r[key_s].values,
            r[key_qt].values,
            r[key_qd].values,
            r[key_ms_first_response].values,
            r[key_attempts].values,
            r['correct'].values
            ))

print(len(group))

28834


In [60]:
from tqdm import tqdm
from sklearn.model_selection import train_test_split

train, test = train_test_split(group, test_size=0.2, random_state=seed)

def generate_df_by_group(group):
    df = pd.DataFrame(columns=['user_id', 'q_idx', 's_idx', 'q_type', 'q_diff', 'ms_first_response', 'attempt_count', 'correct'])
    for user_id, (q, s, qt, qd, ms_first_response, attempts, correct) in tqdm(group.items()):
        for i in range(len(q)):
            tmp = [user_id, q[i], s[i], qt[i], qd[i], ms_first_response[i], attempts[i], correct[i]]
            df.loc[len(df)] = tmp
    return df

df_test = generate_df_by_group(test)
df_train = generate_df_by_group(train)

5767it [1:28:06,  1.09it/s]
23067it [15:17:00,  2.39s/it]


In [61]:
from sklearn.preprocessing import MinMaxScaler

sacler = MinMaxScaler()

tmp = sacler.fit_transform(df_train[[key_ms_first_response, key_attempts]])
df_train[[key_ms_first_response, key_attempts]] = tmp

tmp = sacler.transform(df_test[[key_ms_first_response, key_attempts]])
df_test[[key_ms_first_response, key_attempts]] = tmp

df_train

,user_id,q_idx,s_idx,q_type,q_diff,ms_first_response,attempt_count,correct
0,123347.0,25458.0,67.0,0.0,0.931818,0.000058,0.034483,1.0
1,123347.0,34684.0,60.0,0.0,0.734940,0.000071,0.034483,1.0
2,123347.0,628.0,60.0,0.0,0.565217,0.000063,0.034483,1.0
3,123347.0,16890.0,67.0,0.0,0.904762,0.000175,0.034483,1.0
4,123347.0,3451.0,60.0,0.0,0.689655,0.000087,0.034483,1.0
...,...,...,...,...,...,...,...,...
2116779,212403.0,28725.0,94.0,0.0,0.937500,0.000183,0.034483,1.0
2116780,212403.0,11215.0,122.0,0.0,1.000000,0.000120,0.034483,1.0
2116781,212403.0,18272.0,56.0,0.0,0.600000,0.000173,0.103448,0.0
2116782,212403.0,36096.0,117.0,0.0,1.000000,0.000298,0.034483,1.0


In [62]:
df_train.to_csv("train.csv", index=None)
df_test.to_csv("test.csv", index=None)

### edge

In [63]:
problems = df['problem_id'].unique() # 获取去重的问题序列
pro_id_dict = dict(zip(problems, range(len(problems)))) # 问题id：index
print('problem number %d' % len(problems))

# skill_id_dict, skill_cnt = {}, 0
skills = df['skill'].unique()
skill_id_dict = dict(zip(skills, range(len(skills))))

file = open(os.path.join(os.getcwd(), 'edge_list.dat'), 'w')
file.write("{}\t{}\t{}\n".format('qid', 'sid', 'weight'))
for pro_id in tqdm(range(len(problems))): 
    tmp_df = df[df['problem_id']==problems[pro_id]]
    tmp_df_0 = tmp_df.iloc[0]  # problem_id
    # build problem-skill bipartite
    file.write("{}\t{}\t{}\n".format(pro_id, skill_id_dict[tmp_df_0['skill']], 1))

# beacuse the padding of q and s is num_q+1 and num_s+1
file.write("{}\t{}\t{}\n".format(len(problems), len(skill_id_dict), 1))
file.close()

problem number 50988


100%|██████████| 50988/50988 [01:58<00:00, 432.02it/s]
